In [1]:
# Install Transformer library
!pip install transformers torch

In [2]:
#importing mmodel using transformer
from transformers import pipeline

In [3]:
chatbot = pipeline(
    "text-generation",
    model="mistralai/Mistral-7B-Instruct-v0.1",
    pad_token_id=2,
    max_new_tokens=300
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [4]:
#prompt engineering to make the responses friendly and clear
SYSTEM_PROMPT = """
You are a helpful medical assistant chatbot. Act like a top 1% helpful medical assistants. Give simple and friendly health information.

Safety Rules:
- Do not diagnose diseases.
- Do not prescribe medicines.
- Do not give exact dosage or overdose limits.
- For medicine questions, do not simply say "yes". Say it may be used only as directed by a doctor/pharmacist or medicine label.
- For children, pregnancy, allergies, chronic illness, or serious symptoms, recommend consulting a doctor or pharmacist.
"""

danger_keywords = [
    "chest pain",
    "difficulty breathing",
    "suicide",
    "overdose",
    "severe bleeding"
]

In [5]:
#create safety filter function
def safety_filter(query):
    query = query.lower()
    for word in danger_keywords:
        if word in query:
            return True
    return False

In [6]:
def health_chatbot(user_query):

    if safety_filter(user_query):
        return "This may be serious. Please contact a doctor or emergency service immediately."

    prompt = f"""
    {SYSTEM_PROMPT}

    User: {user_query}

    Assistant:
    """

    response = chatbot(
        prompt,
        temperature=0.5,
        do_sample=True,
        return_full_text=False,
        eos_token_id=2
    )

    answer = response[0]["generated_text"].strip()

    # Remove extra generated chat
    stop_words = ["User:", "Assistant:"]

    for word in stop_words:
        if word in answer:
            answer = answer.split(word)[0]

    return answer.strip()

In [12]:
# Create chatbot function
def health_chatbot(user_query):
  if safety_filter(user_query):
        return "This may be a serious medical emergency. Please contact a doctor or emergency service immediately."

  prompt = SYSTEM_PROMPT + "\nUser: " + user_query + "\nAssistant:"

  response = chatbot(
        prompt,
        do_sample=True,
        temperature=0.7,
        return_full_text=False
    )

  answer = response[0]["generated_text"].strip()

  return answer


In [13]:
# Test Questions
questions = [
    "What causes a sore throat?",
    "Is paracetamol safe for children?"
]

for q in questions:

    print("\nUser:", q)
    print("Bot:", health_chatbot(q))
    print("-" * 50)


User: What causes a sore throat?
Bot: A sore throat can be caused by a variety of factors such as viral infections like the common cold or flu, bacterial infections, allergies, or even irritation from smoking or throat irritants. If you're experiencing a sore throat, it's important to consult with a healthcare professional to determine the cause and appropriate treatment.
--------------------------------------------------

User: Is paracetamol safe for children?
Bot: Yes, paracetamol may be used for children as directed by a doctor or medicine label. However, if your child has a fever, it's important to consult a doctor first.
--------------------------------------------------


In [14]:
#to take real time input and test model
print("Health Assistant Chatbot")
print("Type your health question below. Type 'quit' to exit.")

user_query = ""

while user_query != "q":

    user_query = input("\n Hey Wassup please let us know how i can assist you: ")

    if user_query == "q":
        print("Thanks for choosing General Health Query Chatbot")
        break

    response = health_chatbot(user_query)
    print("\n \n Bot:", response)


Health Assistant Chatbot
Type your health question below. Type 'quit' to exit.

 Hey Wassup please let us know how i can assist you: What causes a sore throat?

 
 Bot: A sore throat can be caused by many different things, like a virus or bacteria. It's important to rest and drink plenty of fluids to help alleviate symptoms. If symptoms persist or worsen, it may be best to see a doctor.

 Hey Wassup please let us know how i can assist you: Is paracetamol safe for children?

 
 Bot: Paracetamol may be used for children as directed by a healthcare provider or medicine label. However, it's important to follow the correct dosage and consult a doctor or pharmacist if you have any concerns or questions about its use.

 Hey Wassup please let us know how i can assist you: How much water should I drink per day?

 
 Bot: It's recommended to drink at least 8 cups of water per day. However, the ideal amount varies depending on individual factors like age, weight, and activity level. It's always be